In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
# --- 1. SETUP AND CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 128

from google.colab import drive
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/value-vortex/'
DATA_PATH = os.path.join(PROJECT_PATH, 'dataset/')
IMAGE_PATH = os.path.join(PROJECT_PATH, 'images/all_images/')
FEATURES_PATH = os.path.join(PROJECT_PATH, 'features/')
os.makedirs(FEATURES_PATH, exist_ok=True)

print(f"Using device: {DEVICE}")
if not torch.cuda.is_available():
    print("WARNING: GPU NOT DETECTED. PLEASE ENABLE GPU RUNTIME.")

Mounted at /content/drive
Using device: cuda


In [3]:
# --- 2. LOAD MODEL AND TRANSFORMS ---
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier = nn.Identity()
model.to(DEVICE)
model.eval()

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("Model and transforms are ready.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 171MB/s]


Model and transforms are ready.


In [4]:
# --- 3. LOAD THE FIRST HALF OF THE TEST DATA ---
print("\nLoading the first half of the test set...")
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))
half_point = len(test_df) // 2
test_df_part1 = test_df.head(half_point)

ids_to_process = test_df_part1['sample_id'].tolist()
image_files_to_process = [f"{sid}.jpg" for sid in ids_to_process]

print(f"Total test images to process: {len(ids_to_process)}")



Loading the first half of the test set...
Total test images to process: 37500


In [ ]:
# Load the pre-trained EfficientNet-B0 model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# CRITICAL: We remove the final layer (the classifier)
# We want the features from the layer before it.
model.classifier = nn.Identity()

# Move the model to the GPU and set it to evaluation mode
model.to(DEVICE)
model.eval()

print("EfficientNet-B0 model loaded and configured for feature extraction.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 139MB/s]


EfficientNet-B0 model loaded and configured for feature extraction.


In [6]:
# --- 4. THE FEATURE EXTRACTION LOOP ---
all_embeddings = {}

if ids_to_process:
    with torch.no_grad():
        for i in tqdm(range(0, len(image_files_to_process), BATCH_SIZE)):
            batch_files = image_files_to_process[i:i+BATCH_SIZE]
            batch_tensors = []

            for img_file in batch_files:
                img_path = os.path.join(IMAGE_PATH, img_file)
                if os.path.exists(img_path):
                    try:
                        image = Image.open(img_path).convert('RGB')
                        tensor = transform(image)
                        batch_tensors.append(tensor)
                    except Exception:
                        batch_tensors.append(torch.zeros((3, IMG_SIZE, IMG_SIZE)))
                else:
                    batch_tensors.append(torch.zeros((3, IMG_SIZE, IMG_SIZE)))

            if not batch_tensors: continue

            batch = torch.stack(batch_tensors).to(DEVICE)
            features = model(batch)
            features_cpu = features.cpu().numpy()

            for j, file_name in enumerate(batch_files):
                sample_id = int(file_name.split('.')[0])
                all_embeddings[sample_id] = features_cpu[j]

 23%|██▎       | 68/293 [56:57<3:08:27, 50.25s/it]


KeyboardInterrupt: 